In [ ]:

## Packages ---

import numpy as np
import pandas as pd
import geopandas as gpd
from arcgis.features import GeoAccessor, GeoSeriesAccessor
import getpass
from pathlib import Path
from tqdm import tqdm
from datetime import date
from IPython.display import display


## File paths ---

user = getpass.getuser()
path_users = Path.home()

path_sp = path_users / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents'
path_prod = path_sp / 'Products'
path_cap = path_prod / 'Cap to Cap'
path_arcpro = Path(r"I:\Projects\Josh\Regional Monitoring\ArcPro_sup\Cap to Cap")


export=False



In [ ]:


## MSA Level ACS data ---

file_cap1_bg = path_cap / 'original_data' / 'Cap_1 MSA ACS5.xlsx'
file_cap2_bg = path_cap / 'original_data' / 'Cap_2 MSA ACS5.xlsx'
sheet_name_bg = 'MSA'

df_cap1_msa = pd.read_excel(file_cap1_bg, sheet_name=sheet_name_bg)
df_cap2_msa = pd.read_excel(file_cap2_bg, sheet_name=sheet_name_bg)


df_cap1_msa = df_cap1_msa[df_cap1_msa['Race_Ethnicity'] == 'All']
df_cap1_msa = df_cap1_msa[['MSA_ID', 'MSA', 'Total']]
df_cap1_msa = df_cap1_msa.rename(columns={'Total':'Population'})
df_cap1_msa = df_cap1_msa.reset_index(drop=True)

df_cap2_msa = df_cap2_msa[['MSA_ID', 'MSA', 'Total']]
df_cap2_msa = df_cap2_msa.rename(columns={'Total':'Housing Units'})

df_msa = df_cap1_msa.merge(df_cap2_msa, on=['MSA_ID', 'MSA'])

display(df_msa)



In [ ]:

path_gdb = path_arcpro / 'CapToCap.gdb'

file_rosslyn             = path_gdb / 'GIS_open_neighborhoods_VA_rosslyn_corridor_Enrich'
file_columbia            = path_gdb / 'GIS_open_neighborhoods_VA_columbia_pike_corridor_Enrich'
file_sac_gold            = path_gdb / 'SACOG_neighborhoods_gold_line_station_Enrich'
file_sac_downtown        = path_gdb / 'SACOG_neighborhoods_downtown_sac_Enrich'
file_sac_west_broadway   = path_gdb / 'SACOG_neighborhoods_sac_west_broadway_Enrich'
file_sac_green           = path_gdb / 'SACOG_neighborhoods_sac_future_green_line_Enrich'
file_sac_railyards       = path_gdb / 'SACOG_neighborhoods_sac_railyards_Enrich'
file_sac_arden_west_expo = path_gdb / 'SACOG_neighborhoods_sac_arden_west_expo_Enrich'
file_sac_natomasI5       = path_gdb / 'SACOG_neighborhoods_sac_natomas_I5_Enrich'


sdf_rosslyn             = pd.DataFrame.spatial.from_featureclass(file_rosslyn            )
sdf_columbia            = pd.DataFrame.spatial.from_featureclass(file_columbia           )
sdf_sac_gold            = pd.DataFrame.spatial.from_featureclass(file_sac_gold           )
sdf_sac_downtown        = pd.DataFrame.spatial.from_featureclass(file_sac_downtown       )
sdf_sac_west_broadway   = pd.DataFrame.spatial.from_featureclass(file_sac_west_broadway  )
sdf_sac_green           = pd.DataFrame.spatial.from_featureclass(file_sac_green          )
sdf_sac_railyards       = pd.DataFrame.spatial.from_featureclass(file_sac_railyards      )
sdf_sac_arden_west_expo = pd.DataFrame.spatial.from_featureclass(file_sac_arden_west_expo)
sdf_sac_natomasI5       = pd.DataFrame.spatial.from_featureclass(file_sac_natomasI5      )


sdf_rosslyn            ['File'] = 'GIS_open_neighborhoods_VA_rosslyn_corridor_Enrich'
sdf_columbia           ['File'] = 'GIS_open_neighborhoods_VA_columbia_pike_corridor_Enrich'
sdf_sac_gold           ['File'] = 'SACOG_neighborhoods_gold_line_station_Enrich'
sdf_sac_downtown       ['File'] = 'SACOG_neighborhoods_downtown_sac_Enrich'
sdf_sac_west_broadway  ['File'] = 'SACOG_neighborhoods_sac_west_broadway_Enrich'
sdf_sac_green          ['File'] = 'SACOG_neighborhoods_sac_future_green_line_Enrich'
sdf_sac_railyards      ['File'] = 'SACOG_neighborhoods_sac_railyards_Enrich'
sdf_sac_arden_west_expo['File'] = 'SACOG_neighborhoods_sac_arden_west_expo_Enrich'
sdf_sac_natomasI5      ['File'] = 'SACOG_neighborhoods_sac_natomas_I5_Enrich'


sdf_neighborhoods = pd.concat([sdf_rosslyn, sdf_columbia])
sdf_neighborhoods = sdf_neighborhoods.rename(columns = {'NEIGHBORHO':'Neighborhood', 'householdtotals_TOTHH_CY':'Households'})
df_neighborhoods = sdf_neighborhoods[['File', 'Neighborhood', 'Households']]

df_neighborhoods.loc[df_neighborhoods['Neighborhood'] == 'Ballston - Virginia Square', 'Neighborhood'] = 'Ballston-Virginia Square'
df_neighborhoods.loc[df_neighborhoods['Neighborhood'] == 'Clarendon - Courthouse'    , 'Neighborhood'] = 'Clarendon-Courthouse'
df_neighborhoods.loc[df_neighborhoods['Neighborhood'] == 'Radnor/Fort Myer Heights'  , 'Neighborhood'] = 'Radnor-Ft Myer Heights'

sdf_neighborhoods2 = pd.concat([sdf_sac_gold, sdf_sac_downtown, sdf_sac_west_broadway, sdf_sac_green, sdf_sac_railyards, sdf_sac_arden_west_expo, sdf_sac_natomasI5])
sdf_neighborhoods2 = sdf_neighborhoods2.rename(columns = {'NAME':'Neighborhood', 'householdtotals_TOTHH_CY':'Households'})
df_neighborhoods2 = sdf_neighborhoods2[['File', 'Neighborhood', 'Households']]
df_neighborhoods2.loc[df_neighborhoods2['Neighborhood'] == 'College/Glen', 'Neighborhood'] = 'College-Glen'
df_neighborhoods2.loc[df_neighborhoods2['Neighborhood'] == 'Med Center', 'Neighborhood'] = 'Medical Center'


df_neighborhoods = pd.concat([df_neighborhoods, df_neighborhoods2, df_neighborhoods2[df_neighborhoods2['Neighborhood'] == 'Old Sacramento']])
df_neighborhoods = df_neighborhoods.sort_values(['File', 'Neighborhood'])
df_neighborhoods = df_neighborhoods.reset_index(drop=True)
df_neighborhoods.loc[26, 'Neighborhood'] = 'Old North Sacramento'
df_neighborhoods.loc[27, 'Neighborhood'] = 'Old West Sacramento'
df_neighborhoods.loc[26, 'Households'  ] = 44
df_neighborhoods.loc[27, 'Households'  ] = 44
display(df_neighborhoods)


In [ ]:


## Zillow data ----------------------------------------------------------------------------------------------------------------

# Regions

regions_zillow = ["Sacramento, CA", "Yuba City, CA", "Washington, DC"]


file_in = path_cap / 'original_data' / "Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"
df_sales = pd.read_csv(file_in)
df_sales = df_sales[df_sales['RegionType'] == 'msa']
df_sales = pd.melt(df_sales, id_vars=['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName'], var_name='date_', value_name='Price')

df_sales = df_sales[['RegionName', 'StateName', 'date_', 'Price']]
df_sales.columns = ['Region', 'State', 'date_', 'Price']
df_sales = df_sales[df_sales['Region'].isin(regions_zillow)]
df_sales['date_'] = pd.to_datetime(df_sales['date_'])
df_sales['Year'] = df_sales['date_'].dt.year
df_sales = df_sales.drop('date_', axis = 1)
df_sales = df_sales.groupby(['State', 'Region', 'Year'], as_index = False)['Price'].mean()
df_sales1 = df_sales[df_sales['Year'] == 2024]
display(df_sales1)


# Cities

cities_zillow = [
    'Sacramento'
, 'Arlington'
, 'Alexandria'
, 'Elk Grove'
, 'Falls Church'
, 'Rancho Cordova'
, 'Davis'
, 'Citrus Heights'
]

file_in = path_cap / 'original_data' / "City_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"
df_sales = pd.read_csv(file_in)
df_sales = df_sales[df_sales['RegionType'] == 'city']
df_sales = pd.melt(df_sales, id_vars=['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName', 'State', 'Metro', 'CountyName'], var_name='date_', value_name='Price')

df_sales = df_sales[['RegionName', 'StateName', 'date_', 'Price']]
df_sales.columns = ['Jurisdiction', 'State', 'date_', 'Price']
df_sales = df_sales[df_sales['State'].isin(['VA', 'CA'])]
df_sales = df_sales[df_sales['Jurisdiction'].isin(cities_zillow)]
df_sales['date_'] = pd.to_datetime(df_sales['date_'])
df_sales['Year'] = df_sales['date_'].dt.year
df_sales = df_sales.drop('date_', axis = 1)
df_sales = df_sales.groupby(['State', 'Jurisdiction', 'Year'], as_index = False)['Price'].mean()
df_sales2 = df_sales[df_sales['Year'] == 2024]
display(df_sales2)



# Neighborhoods

neighborhoods = [
    'Potomac Yard-Potomac Greens'
        , 'Braddock Road Metro'
        , 'Arlington Mill'
        , 'Arden Arcade'
        , 'Upper Land Park'
]

corridor_columbia_pike = [
    'Arlington Mill'          # and as its own neighborhood
        , 'Penrose'          
        , 'Arlington Heights'
        , 'Alcova Heights'   
        , 'Barcroft'         
        , 'Forest Glen'      
        , 'Columbia Heights' 
        , 'Douglas Park'      # (mostly)
        , 'Columbia Forest'  
        , 'Claremont'
]

corridor_ballston = [
    'North Rosslyn'                 
        , 'Radnor-Ft Myer Heights'  
        , 'Colonial Village'        
        , 'Lyon Village'            
        , 'Clarendon-Courthouse'    
        , 'Ballston-Virginia Square'
        , 'Ashton Heights'           # (somewhat)
        , 'Bluemont'                 # (mostly)
        , 'Glencarlyn'               # (somewhat)
        , 'Buckingham'               # (somewhat)
]

# df_neighborhoods[df_neighborhoods['File'].str.contains('arden')].Neighborhood.values
corridor_gold_line = [
    'East Sacramento'
    # , 'CSUS' ##
    , 'College-Glen' # 
    , 'Alhambra Triangle'
    , 'Elmhurst'
    , 'Medical Center' #
    , 'Tahoe Park'
    , 'Tahoe Park East'
    # , 'Granite Regional Park' ##
    , 'Colonial Manor'
    # , 'Belvedere' ##
    # , 'Ramona Village' ##
    # , 'College Town' ##
]

corridor_downtown_sac = [
    'Alkali Flat'
    , 'Mansion Flats'
    , 'Old Sacramento' # Old North vs Old West
    , 'Downtown'
    , 'Southside Park'
    , 'Richmond Grove'
]

# corridor_railyards = ['Southern Pacific / Richards'] # Nothing in zillow

corridor_green_line = [
    # 'RP - Sports Complex'
    'Village 5'
    , 'Natomas Crossing'
    , 'South Natomas'
    # , 'Natomas Corporate Center'
]

corridor_arden_west_expo = [
    'Swanston Estates'
    # , 'Arden Fair'
    # , 'Point West'
    # , 'Cal Expo'
]

corridor_natomas = [
    'Village 7'
    , 'Natomas Crossing'
    , 'Creekside'
    , 'Gateway Center'
    , 'Gateway West'
    , 'Metro Center'
    , 'Natomas Creek'
    , 'Westlake'
    , 'Willowcreek'
]



neighborhoods = neighborhoods + corridor_columbia_pike + corridor_ballston + corridor_arden_west_expo + corridor_green_line + corridor_downtown_sac + corridor_gold_line
neighborhoods_CA = ['Arden Arcade', 'Upper Land Park'] + corridor_arden_west_expo + corridor_green_line + corridor_downtown_sac + corridor_gold_line
neighborhoods_VA = ['Potomac Yard-Potomac Greens', 'Braddock Road Metro', 'Arlington Mill'] + corridor_columbia_pike + corridor_ballston


file_in = path_cap / 'original_data' / "Neighborhood_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"
df_sales = pd.read_csv(file_in)
df_sales = df_sales[~((df_sales['RegionName'] == 'Willowcreek') & (df_sales['City'] == 'Davis'))]
df_sales = df_sales[df_sales['RegionType'] == 'neighborhood']
df_sales = pd.melt(df_sales, id_vars=['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName', 'State', 'City', 'Metro', 'CountyName'], var_name='date_', value_name='Price')

df_sales = df_sales[['RegionName', 'StateName', 'date_', 'Price']]
df_sales.columns = ['Neighborhood', 'State', 'date_', 'Price']
df_sales = df_sales[
    ((df_sales['State'].isin(['VA'])) & (df_sales['Neighborhood'].isin(neighborhoods_VA))) | ((df_sales['State'].isin(['CA'])) & (df_sales['Neighborhood'].isin(neighborhoods_CA)))
    ]
df_sales_am = df_sales[df_sales['Neighborhood'] == 'Arlington Mill']


df_sales['Location'] = df_sales['Neighborhood'].copy()
df_sales.loc[(df_sales['Neighborhood'].isin(corridor_columbia_pike  )) & (df_sales['State'].isin(['VA'])), 'Location'] = 'Columbia Pike Corridor'
df_sales.loc[(df_sales['Neighborhood'].isin(corridor_ballston       )) & (df_sales['State'].isin(['VA'])), 'Location'] = 'Rosslyn-Ballston Corridor'
df_sales.loc[(df_sales['Neighborhood'].isin(corridor_gold_line      )) & (df_sales['State'].isin(['CA'])), 'Location'] = 'Gold Line Station Corridor'
df_sales.loc[(df_sales['Neighborhood'].isin(corridor_downtown_sac   )) & (df_sales['State'].isin(['CA'])), 'Location'] = 'Downtown Sacramento'
df_sales.loc[(df_sales['Neighborhood'].isin(corridor_green_line     )) & (df_sales['State'].isin(['CA'])), 'Location'] = 'Future Green Line Station Corridor'
df_sales.loc[(df_sales['Neighborhood'].isin(corridor_arden_west_expo)) & (df_sales['State'].isin(['CA'])), 'Location'] = 'Arden-Arcade Corridors'
df_sales.loc[(df_sales['Neighborhood'].isin(corridor_natomas        )) & (df_sales['State'].isin(['CA'])), 'Location'] = 'Natomas I5 Corridor'

df_sales_am['Location'] = 'Arlington Mill'
df_sales = pd.concat([df_sales, df_sales_am])
df_sales = df_sales.merge(df_neighborhoods, on='Neighborhood', how='left')
df_sales.loc[df_sales['Households'].isna(), 'Households'] = 1
df_sales = df_sales.reset_index(drop=True)

wm = lambda x: np.average(x, weights = df_sales.loc[x.index, "Households"]) # weighted average
df_sales['date_'] = pd.to_datetime(df_sales['date_'])
df_sales['Year'] = df_sales['date_'].dt.year
df_sales = df_sales.drop('date_', axis = 1)
df_sales = df_sales.groupby(['State', 'Location', 'Year'], as_index = False).agg(Price = ('Price', wm)) # Needs number of households for neighborhoods to corridors
df_sales = df_sales[df_sales['Year'] == 2024]
df_sales3 = df_sales.reset_index(drop=True)
display(df_sales3)



In [ ]:


# Regions

regions_zillow = ["Sacramento, CA", "Yuba City, CA", "Washington, DC"]


file_in = path_cap / 'original_data' / "Metro_zori_uc_sfrcondomfr_sm_month.csv"
df_rent = pd.read_csv(file_in)
df_rent = df_rent[df_rent['RegionType'] == 'msa']
df_rent = pd.melt(df_rent, id_vars=['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName'], var_name='date_', value_name='Rent')

df_rent = df_rent[['RegionName', 'StateName', 'date_', 'Rent']]
df_rent.columns = ['Region', 'State', 'date_', 'Rent']
df_rent = df_rent[df_rent['Region'].isin(regions_zillow)]
df_rent['date_'] = pd.to_datetime(df_rent['date_'])
df_rent['Year'] = df_rent['date_'].dt.year
df_rent = df_rent.drop('date_', axis = 1)
df_rent = df_rent.groupby(['State', 'Region', 'Year'], as_index = False)['Rent'].mean()
df_rent1 = df_rent[df_rent['Year'] == 2024]
display(df_rent1)


# Cities

cities_zillow = ['Sacramento'
, 'Arlington'
, 'Alexandria'
, 'Citrus Heights'
, 'Elk Grove'
, 'Roseville'
, 'Falls Church'
, 'Rancho Cordova'
, 'Yuba City'
, 'Folsom'
, 'Davis'
, 'Rocklin'
, 'Auburn'
, 'Marysville'
, 'Placerville'
]

file_in = path_cap / 'original_data' / "City_zori_uc_sfrcondomfr_sm_month.csv"
df_rent = pd.read_csv(file_in)
df_rent = df_rent[df_rent['RegionType'] == 'city']
df_rent = pd.melt(df_rent, id_vars=['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName', 'State', 'Metro', 'CountyName'], var_name='date_', value_name='Rent')

df_rent = df_rent[['RegionName', 'StateName', 'date_', 'Rent']]
df_rent.columns = ['Jurisdiction', 'State', 'date_', 'Rent']
df_rent = df_rent[df_rent['State'].isin(['VA', 'CA'])]
df_rent = df_rent[df_rent['Jurisdiction'].isin(cities_zillow)]
df_rent['date_'] = pd.to_datetime(df_rent['date_'])
df_rent['Year'] = df_rent['date_'].dt.year
df_rent = df_rent.drop('date_', axis = 1)
df_rent = df_rent.groupby(['State', 'Jurisdiction', 'Year'], as_index = False)['Rent'].mean()
df_rent2 = df_rent[df_rent['Year'] == 2024]
display(df_rent2)



In [ ]:


if export:
    file_msa = path_cap / 'Analysis' / 'MSA_Population and Housing.xlsx'
    df_msa.to_excel(file_msa, index=False)

    file_corridor = path_cap / 'Analysis' / 'Corridors and Neighborhoods_Housing Sales Prices.xlsx'
    df_sales3.to_excel(file_corridor, index=False)

    file_corridor = path_cap / 'Analysis' / 'CDPs_Housing Sales Prices.xlsx'
    df_sales2.to_excel(file_corridor, index=False)

    file_msa = path_cap / 'Analysis' / 'MSA_Housing Sales Price.xlsx'
    df_sales1.to_excel(file_msa, index=False)


    file_corridor = path_cap / 'Analysis' / 'CDPs_Rent Prices.xlsx'
    df_rent2.to_excel(file_corridor, index=False)

    file_msa = path_cap / 'Analysis' / 'MSA_Rent Price.xlsx'
    df_rent1.to_excel(file_msa, index=False)


